# **PERFORM BATCH SCORING AND SAVE PREDICTIONS TO A LAKEHOUSE**

How import the registered LightGBMClassifier model that it was built in notebook 3 of this current repository.

Fabric allows to opoerationalize ML models with a scalble function called PREDICT.

To generate batch predictions on the test dataset, it will be use Version 1 of the trained LightGBM model that demonstrated the best performance among all trained ML models.

The function PREDICT can be invoked using:
- Transformer API from SypnapseML
- Spark SQL API
- Pyspark user-defined function (UDF)

In [8]:
%pip install "mlflow==2.12.2"

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 20, Finished, Available, Finished, True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 129.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 52.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 32.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 73.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 20.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Not uninstalling typing-extensions at /home/trusted-service-user/cluster-env/trident_env/lib/python3.10/site-packages, outside environment /nfs4/pyenv-5b30beaf-502f-48a2-8e08-984334e0596f
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 2.0.0 requires sentenc

In [1]:
%pip install scikit-learn==1.6.1


StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 7, Finished, Available, Finished, True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 132.9 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.0
    Not uninstalling scikit-learn at /home/trusted-service-user/cluster-env/trident_env/lib/python3.10/site-packages, outside environment /nfs4/pyenv-5b30beaf-502f-48a2-8e08-984334e0596f
    Can't uninstall 'scikit-learn'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lime 0.2.0.1 requires scikit-image>=0.12, which is not installed.
sentence-transformers 2.0.0 requires sentencepiece, which is not installed.
sentence-transformers 2.0.0 requires torchvision, which is not installed.

[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use u

In [2]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

models = client.search_registered_models()

for model in models:
    print(f"Modelo: {model.name}")

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 9, Finished, Available, Finished, False)

Modelo: ML_diabetes_classification
Modelo: rfc1_sm
Modelo: rfc2_sm
Modelo: lgbm_sm


In [3]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

models = client.search_registered_models()

for model in models:
    print(f"\nModelo: {model.name}")
    
    for version in model.latest_versions:
        print(f"  Versión: {version.version}")
        print(f"  Estado: {version.status}")
        print(f"  Run ID: {version.run_id}")

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 10, Finished, Available, Finished, False)


Modelo: ML_diabetes_classification
  Versión: 1
  Estado: READY
  Run ID: 074d3c09-0e7a-4e0b-97e5-13bdb810ea6e

Modelo: rfc1_sm
  Versión: 1
  Estado: READY
  Run ID: f7bea407-3227-464f-a7d7-d79535023d44

Modelo: rfc2_sm
  Versión: 1
  Estado: READY
  Run ID: 76f145eb-6693-4829-991a-0479caa6fab3

Modelo: lgbm_sm
  Versión: 1
  Estado: READY
  Run ID: 239ae48b-7773-4244-a7e1-65750e268116


## LOAD TEST DATA

In [11]:
df_test = spark.read.format("delta").load("Tables/df_test")
display(df_test)

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bdb8151d-d843-4f4e-ba41-14562550abba)

## PREDICT WITH TRANSFORMER  API

To use Transformer API, it must first create an MLFlowTransformer object.

This transformer serves as wrapper around  the model that it was created in notebook 3. It allows to generate batch predictions on a given dataFrame. It is necessary provide the following parameters:
- The test dataframe columns that the model needs as an input.
- A name for the new output column
- The correct model name and model version.

In [19]:
from synapse.ml.predict import MLFlowTransformer

model = MLFlowTransformer(
    inputCols=list(df_test.columns),
    outputCol='predictions',
    modelName='lgbm_sm',
    modelVersion=1
)

predictions = model.transform(df_test)

display(predictions)

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 34, Finished, Available, Finished, False)

2026/05/31 15:44:01 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - numpy (current: 1.24.3, required: numpy==1.26.4)
 - scipy (current: 1.10.1, required: scipy==1.15.3)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


SynapseWidget(Synapse.DataFrame, ba29bdd1-a355-48a6-86a3-bca1f7ba7889)

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 35, Finished, Available, Finished, False)

Now we have the MLFlowTransformer object, it is possible to generate batch predictions:

## PREDICT with SPARK SQL API

In [21]:
from pyspark.ml.feature import SQLTransformer

model_name = 'lgbm_sm'
model_version = 1
features = df_test.columns

sqlt = SQLTransformer().setStatement(
    f"SELECT PREDICT('{model_name}/{model_version}',{','.join(features)}) as predictions FROM __THIS__")
display(sqlt.transform(df_test))

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3f9d399-71b8-4f4a-a773-f2852d977aec)

In [22]:
display(predictions)

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7432fa38-ad20-4125-9a6f-53396b7736f7)

## WRITE MODEL PREDICTION RESULTS TO THE LAKEHOUSE

In [23]:
table_name = "df_test_with_predictions_v1"
predictions.write.format('delta').mode("overwrite").save(f"Tables/{table_name}")
print(f"Spark dataframe saved to delta table: {table_name}")

StatementMeta(, 081b0868-2340-4816-8893-b613cb2696d5, 39, Finished, Available, Finished, False)

Spark dataframe saved to delta table: df_test_with_predictions_v1
